# Despesas CEAPS - Senado/Brasil

Distribui a CEAPS pelo mapa histórico da 57ª legislatura.

Um código só entra em uma pasta estadual quando existe no mapa de senadores que realmente
exerceram mandato na 57ª legislatura. Registros sem correspondência não são apagados:
ficam em `_despesas_nao_distribuidas_YYYY.csv` para auditoria.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import tempfile
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOGO = "workspace"
SCHEMA = "pi_ii_bronze"
VOLUME = "senado"
LEGISLATURA = 57
ANOS = [2023, 2024, 2025, 2026]

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

ROOT = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}")
TMP = Path(tempfile.mkdtemp(prefix="pi_ii_bronze_senado_v2_"))

LEGIS_BASE = "https://legis.senado.leg.br/dadosabertos"
ADM_BASE = "https://adm.senado.gov.br/adm-dadosabertos"

retry = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "PI-II-Univesp-Bronze-Senado-Brasil/2.0",
})
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def pasta_uf(uf):
    return ROOT / uf.lower()

def normalizar_nome(valor):
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())

def lista(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def api_get(url, params=None, timeout=(30, 180)):
    r = session.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def extrair_por_chave(obj, chave):
    alvo = normalizar_nome(chave)
    encontrados = []

    def walk(x):
        if isinstance(x, dict):
            for k, v in x.items():
                if normalizar_nome(k) == alvo:
                    if isinstance(v, list):
                        encontrados.extend(
                            [i for i in v if isinstance(i, dict)]
                        )
                    elif isinstance(v, dict):
                        encontrados.append(v)
                walk(v)
        elif isinstance(x, list):
            for item in x:
                walk(item)

    walk(obj)

    unicos = []
    vistos = set()

    for item in encontrados:
        canon = json.dumps(
            item,
            ensure_ascii=False,
            sort_keys=True,
            default=str,
        )
        if canon not in vistos:
            vistos.add(canon)
            unicos.append(item)

    return unicos

def maior_lista_de_dicts(obj):
    listas = []

    def walk(x):
        if isinstance(x, list):
            if x and all(isinstance(i, dict) for i in x):
                listas.append(x)
            for item in x:
                walk(item)
        elif isinstance(x, dict):
            for v in x.values():
                walk(v)

    walk(obj)

    return max(listas, key=len) if listas else []

def achar_coluna(df, candidatos=None, contem_todos=None):
    candidatos = candidatos or []
    mapa = {normalizar_nome(c): c for c in df.columns}

    for nome in candidatos:
        chave = normalizar_nome(nome)
        if chave in mapa:
            return mapa[chave]

    if contem_todos:
        termos = [normalizar_nome(x) for x in contem_todos]
        for c in df.columns:
            nc = normalizar_nome(c)
            if all(t in nc for t in termos):
                return c

    return None

def detectar_ano(df):
    candidatos = [
        c for c in df.columns
        if normalizar_nome(c) in {
            "anomateria", "materiaano", "anoproposicao", "ano"
        }
        or normalizar_nome(c).endswith("anomateria")
    ]

    for c in candidatos:
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().any():
            return s.astype("Int64")

    datas = [
        c for c in df.columns
        if "data" in normalizar_nome(c)
    ]

    for c in datas:
        dt = pd.to_datetime(
            df[c],
            errors="coerce",
            dayfirst=True,
        )
        if dt.notna().any():
            return dt.dt.year.astype("Int64")

    return pd.Series(
        [pd.NA] * len(df),
        index=df.index,
        dtype="Int64",
    )

def carregar_exercicios():
    path = ROOT / "_senadores_exercicios_brasil.csv"

    if not path.exists():
        raise FileNotFoundError(
            "Execute primeiro o notebook 00 v2."
        )

    df = pd.read_csv(
        path,
        sep=";",
        dtype=str,
        keep_default_na=False,
    )

    if df.empty:
        raise RuntimeError("Mapa histórico de exercícios está vazio.")

    for c in ["data_inicio_exercicio", "data_fim_exercicio"]:
        df[c] = pd.to_datetime(
            df[c],
            errors="coerce",
        )

    return df

def salvar_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
    )
    print(f"Salvo: {path} | {len(df):,}")

def salvar_manifesto(nome, payload):
    payload = dict(payload)
    payload["gerado_em_utc"] = utc_now()

    path = ROOT / f"_manifest_{nome}_brasil.json"

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    print("Manifesto:", path)

print("Bronze Senado:", ROOT)


In [ ]:
exercicios = carregar_exercicios()

mapa_cod_uf = (
    exercicios[
        ["codigo_parlamentar", "uf"]
    ]
    .drop_duplicates()
)

# em um mesmo mandato do Senado, o código deve apontar para uma UF
ambiguos = (
    mapa_cod_uf
    .groupby("codigo_parlamentar")["uf"]
    .nunique()
)

ambiguos = ambiguos[ambiguos > 1]

if len(ambiguos):
    raise RuntimeError(
        "Há códigos parlamentares ligados a mais de uma UF: "
        f"{ambiguos.to_dict()}"
    )

cod_para_uf = (
    mapa_cod_uf
    .drop_duplicates("codigo_parlamentar")
    .set_index("codigo_parlamentar")["uf"]
    .to_dict()
)

print("Códigos históricos no mapa:", len(cod_para_uf))


In [ ]:
resumo = []

for ano in ANOS:
    print("\n" + "=" * 70)
    print("ANO", ano)

    url = f"{ADM_BASE}/api/v1/senadores/despesas_ceaps/{ano}"
    payload = api_get(url)

    registros = maior_lista_de_dicts(payload)

    if not registros and isinstance(payload, list):
        registros = [
            x for x in payload
            if isinstance(x, dict)
        ]

    if not registros:
        raise RuntimeError(
            f"Nenhuma despesa encontrada no endpoint CEAPS de {ano}."
        )

    df = pd.json_normalize(registros, sep="_")

    cod_desp_col = achar_coluna(
        df,
        ["codSenador", "codigoSenador", "CodigoParlamentar"],
        contem_todos=["cod", "senador"],
    )

    if not cod_desp_col:
        raise RuntimeError(
            f"Não encontrei o código do senador nas despesas de {ano}. "
            f"Colunas: {list(df.columns)}"
        )

    df["_codigo_senador_norm"] = (
        df[cod_desp_col]
        .astype(str)
        .str.strip()
    )

    df["_uf_recorte"] = (
        df["_codigo_senador_norm"]
        .map(cod_para_uf)
        .fillna("")
    )

    contagens = {}

    for uf in UFS:
        recorte = df[
            df["_uf_recorte"].eq(uf)
        ].copy()

        salvar_csv(
            recorte,
            pasta_uf(uf) / f"despesas_{ano}.csv",
        )

        contagens[uf] = len(recorte)

    nao_distribuidas = df[
        ~df["_uf_recorte"].isin(UFS)
    ].copy()

    nao_distribuidas["_motivo_auditoria"] = (
        "codigo_senador_fora_do_mapa_historico_leg57"
    )

    salvar_csv(
        nao_distribuidas,
        ROOT / f"_despesas_nao_distribuidas_{ano}.csv",
    )

    codigos_nao_distribuidos = sorted(
        nao_distribuidas["_codigo_senador_norm"]
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .unique()
        .tolist()
    )

    resumo.append({
        "ano": ano,
        "status": (
            "ok"
            if len(nao_distribuidas) == 0
            else "revisar"
        ),
        "registros_brasil": len(df),
        "registros_distribuidos": sum(contagens.values()),
        "registros_nao_distribuidos": len(nao_distribuidas),
        "codigos_nao_distribuidos": codigos_nao_distribuidos,
        "por_uf": contagens,
    })

    print(
        "Brasil:", len(df),
        "| distribuídos:", sum(contagens.values()),
        "| sem mapa histórico:", len(nao_distribuidas),
    )

    if codigos_nao_distribuidos:
        print(
            "Códigos sem mapa:",
            codigos_nao_distribuidos,
        )

salvar_manifesto("despesas", {"resumo": resumo})

display(pd.DataFrame([
    {
        "ano": x["ano"],
        "status": x["status"],
        "registros_brasil": x["registros_brasil"],
        "registros_distribuidos": x["registros_distribuidos"],
        "registros_nao_distribuidos": x["registros_nao_distribuidos"],
    }
    for x in resumo
]))
